<a href="https://colab.research.google.com/github/WwwwNB/Colab_files/blob/main/5243_GNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch_geometric
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import MessagePassing, global_mean_pool
from torch_geometric.utils import add_self_loops
import numpy as np

# Example crystal structures (5 materials)
crystal_structures = [
    {"atoms": [14, 8, 8], "bonds": [(0, 1, 1.6), (0, 2, 1.6)]},   # Si02
    {"atoms": [13, 13, 8, 8, 8], "bonds": [(0, 2, 1.9), (1, 3, 1.9), (0, 4, 1.9)]},  #Al2O3
    {"atoms": [26, 8], "bonds": [(0, 1, 2.1)]},  # FeO
    {"atoms": [30, 16], "bonds": [(0, 1, 2.3)]}, # Zns
    {"atoms": [11, 17], "bonds": [(0, 1, 2.5)]},  # NaCL
]

# Atomic properties (node features)
atomic_properties = {
    14: [1.90, 4], #Si
    8: [3.44, 6], # O
    13: [1.61, 3], #Al
    26: [1.83, 8], #Fe
    30: [1.65, 2], #Zn
    16: [2.58, 6], #S
    11: [0.93, 1], #Na
    17: [3.16, 7] #Cl
}

# create graph data objects
graph_list = []

for crystal in crystal_structures:
    atoms = crystal[ "atoms"]
    bonds = crystal[ "bonds"]
    # Node Features: Electroneg + Valence Electrons
    node_features = torch.tensor([atomic_properties[a] for a in atoms], dtype=torch.float)
    # Edge List (from bonds)
    edge_index = torch.tensor([[b[0], b[1]] for b in bonds] + [[b[1], b[0]] for b in bonds], dtype=torch.long) .t()
    # Edge Features: Bond Lengths
    edge_features = torch.tensor([[b[2]] for b in bonds] * 2, dtype=torch.float)
    # Create PyG Data object
    data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_features, y=torch.tensor([0.0]))  # Dummy target
    graph_list. append (data)

# Convert to batch for training
batched_data = Batch.from_data_list(graph_list)

# Define Message Passing GNN with Post-Gnn MLP
class CrystalGNN(torch.nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, output_dim):
        super(CrystalGNN, self).__init__()

        # stack multiple MP layers
        self.conv1 = MessagePassingLayer(node_dim, edge_dim, hidden_dim)
        self.conv2 = MessagePassingLayer(hidden_dim, edge_dim, hidden_dim)
        self.conv3 = MessagePassingLayer(hidden_dim, edge_dim, hidden_dim)

        # post-GNN MLP
        self.fc1 = torch.nn.Linear(hidden_dim, 16)
        self.fc2 = torch.nn.Linear(16, 8)
        self.fc3 = torch.nn.Linear(8, 1)

    def forward(self, x, edge_index, edge_attr, batch):
       x = self.conv1(x, edge_index, edge_attr)
       x = self.conv2(x, edge_index, edge_attr)
       x = self.conv3(x, edge_index, edge_attr)
       x = global_mean_pool(x, batch)
       x = F.relu(self.fc1(x))
       x = F.relu(self.fc2(x))
       x = self.fc3(x)
       return x


class MessagePassingLayer(MessagePassing):
    def __init__(self, node_dim, edge_dim, hidden_dim):
        super(MessagePassingLayer, self).__init__(aggr='mean')
        self.node_mlp = torch.nn.Linear(node_dim, hidden_dim)
        self.edge_mlp = torch.nn.Linear(edge_dim, hidden_dim)
        self.update_mlp = torch.nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, edge_index, edge_attr):
        # Add self-loops to the adjacency matrix
        edge_index = add_self_loops(edge_index, num_nodes=x.size(0))[0]
        return self.propagate(edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_j, edge_attr):
        # x_i: source node, x_j: target node, edge_attr: edge feature
        return x_j + self.edge_mlp(edge_attr)

    def update(self, aggr_out):
        return self.update_mlp(F.relu(aggr_out))


# initialize model
model = CrystalGNN(node_dim=2, edge_dim=1, hidden_dim=8, output_dim=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.MSELoss()

# Dummy Regression Targets(e.g., Formation Energy)
targets = torch.tensor([[-3.5], [-2.1], [-4.7], [-1.8], [-3.0]])

# Train GNN (10 epochs)
for epoch in range(10):
    optimizer.zero_grad()
    output = model(batched_data.x, batched_data.edge_index, batched_data.edge_attr, batched_data.batch)
    loss = criterion(output, targets) # compare prediction with real material properties
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

# Display Predictions
print("\nPredicted Material Properties (Formation energy):")
print(output.detach().numpy())

RuntimeError: The size of tensor a (2) must match the size of tensor b (8) at non-singleton dimension 1